# UNet++ inference + DenseCRF experiments

Two independent phases -- run Phase 1 once, then iterate on Phase 2 freely.

**Phase 1 (run once):** Load any trained checkpoint, run inference on the test set, save softmax probability arrays to disk.  
**Phase 2 (run repeatedly):** Load saved probs, apply DenseCRF with different parameter combinations, compare metrics.

Data: `/mnt/research/v.gomezgilyaspik/students/smishra/S2-iceberg-areas/train_validate_test_v2/train_validate_test/`  
Environment: `iceberg-unet`

Notes on Shibali's training setup (from train.py):
- No per-band normalization. Model takes raw float32 chips in [0, 1].
- Loss: Dice + CrossEntropy with inverse-frequency class weights.
- IoU: mean over classes 1-2 only (iceberg + shadow), skips class 0 (ocean).
- Masks stored as (N, 1, H, W) int64 in the Dataset, squeezed before loss.
- Augmented checkpoint: val IoU 0.4398, test IoU 0.4138 (39 epochs).

## 0. Setup

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

In [ ]:
# --- paths ---
DATA_DIR = '/mnt/research/v.gomezgilyaspik/students/smishra/S2-iceberg-areas/train_validate_test_v2/train_validate_test/'

# point this at whichever checkpoint you want to evaluate
CHECKPOINT = '/mnt/research/v.gomezgilyaspik/students/smishra/S2-iceberg-areas/runs/s2_20260227_231556/best_model.pth'

# output dir for saved arrays (next to the checkpoint)
OUT_DIR = os.path.dirname(CHECKPOINT)
PROBS_PATH  = os.path.join(OUT_DIR, 'test_probs.npy')
LABELS_PATH = os.path.join(OUT_DIR, 'test_labels.npy')
CHIPS_PATH  = os.path.join(OUT_DIR, 'test_chips.npy')

print('Checkpoint:', CHECKPOINT)
print('Output dir:', OUT_DIR)

## Phase 1 -- Inference (run once per checkpoint)

Loads the test set, runs forward pass, saves softmax probs + labels to disk.  
Skip to Phase 2 if `.npy` files already exist for this checkpoint.

In [ ]:
if os.path.exists(PROBS_PATH):
    print('Prob file already exists -- skip to Phase 2.')
else:
    print('No saved probs found -- will run inference below.')

In [ ]:
if not os.path.exists(PROBS_PATH):

    # 1. load test data
    def load_pkl(path):
        with open(path, 'rb') as f:
            return pickle.load(f)

    X_test = load_pkl(os.path.join(DATA_DIR, 'x_test.pkl'))
    Y_test = load_pkl(os.path.join(DATA_DIR, 'y_test.pkl'))

    # handle mask shape: if (N, 1, H, W), squeeze to (N, H, W)
    if Y_test.ndim == 4 and Y_test.shape[1] == 1:
        Y_test = Y_test[:, 0]

    print(f'Test: {X_test.shape}, masks: {Y_test.shape}')
    print(f'Chip range: [{X_test.min():.3f}, {X_test.max():.3f}]')
    print(f'Mask classes: {np.unique(Y_test)}')

    # 2. load model -- no normalization, raw float32 chips in [0,1]
    model = smp.UnetPlusPlus(
        encoder_name='resnet34',
        encoder_weights=None,
        in_channels=3,
        classes=3,
    )
    model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
    model = model.to(device)
    model.eval()
    print('Checkpoint loaded')

    # 3. inference in batches
    BATCH = 16
    all_probs = []

    with torch.no_grad():
        for start in range(0, len(X_test), BATCH):
            batch = torch.from_numpy(X_test[start:start+BATCH].astype(np.float32)).to(device)
            logits = model(batch)
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)

    all_probs = np.concatenate(all_probs)  # (N, 3, 256, 256)

    # 4. save to disk
    np.save(PROBS_PATH,  all_probs)
    np.save(LABELS_PATH, Y_test)
    np.save(CHIPS_PATH,  X_test)

    print(f'Saved probs  -> {PROBS_PATH}  ({all_probs.nbytes / 1e6:.0f} MB)')
    print(f'Saved labels -> {LABELS_PATH}')
    print(f'Saved chips  -> {CHIPS_PATH}')

## Phase 2 -- DenseCRF experiments

Loads saved probs from disk. Edit `param_sets` and re-run from here.  
No GPU needed, no model reload.

In [ ]:
# load saved arrays
all_probs  = np.load(PROBS_PATH)   # (N, 3, 256, 256)
all_labels = np.load(LABELS_PATH)  # (N, 256, 256)
X_test     = np.load(CHIPS_PATH)   # (N, 3, 256, 256)

baseline_preds = all_probs.argmax(axis=1)  # (N, 256, 256)

print(f'{len(all_probs)} test chips loaded')

In [ ]:
import pydensecrf.densecrf as dcrf
from pydensecrf.utils import unary_from_softmax


def scale_to_uint8(chip):
    """Per-band 2-98 percentile stretch to uint8 for CRF bilateral kernel."""
    out = np.zeros_like(chip, dtype=np.uint8)
    for b in range(chip.shape[0]):
        band  = chip[b]
        valid = band[band > 0]
        if valid.size == 0:
            continue
        lo, hi = np.percentile(valid, [2, 98])
        if hi <= lo:
            hi = lo + 1e-6
        out[b] = np.clip((band - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)
    return out


def apply_crf(prob, chip, sxy_gauss, compat_gauss,
              sxy_bilateral, srgb_bilateral, compat_bilateral, n_iters):
    """
    Apply DenseCRF to one chip.

    prob : (3, H, W) float32 softmax probs
    chip : (3, H, W) float32 raw chip -- scaled to uint8 internally
    Returns (H, W) uint8 class labels.
    """
    n_classes, H, W = prob.shape
    chip_u8 = scale_to_uint8(chip)

    d = dcrf.DenseCRF2D(W, H, n_classes)
    d.setUnaryEnergy(unary_from_softmax(prob))
    d.addPairwiseGaussian(sxy=sxy_gauss, compat=compat_gauss)
    d.addPairwiseBilateral(
        sxy=sxy_bilateral, srgb=srgb_bilateral,
        rgbim=np.ascontiguousarray(chip_u8.transpose(1, 2, 0)),
        compat=compat_bilateral
    )
    Q = d.inference(n_iters)
    return np.argmax(Q, axis=0).reshape(H, W).astype(np.uint8)


def run_crf_batch(probs, chips, params, subset=None):
    """
    Run CRF on chips. If subset is an int, only process that many chips
    (for fast parameter sweeps). Returns (N, H, W) predictions.
    """
    n = len(probs) if subset is None else min(subset, len(probs))
    out = np.zeros((n, 256, 256), dtype=np.uint8)
    for i in range(n):
        out[i] = apply_crf(probs[i], chips[i], **params)
    return out

In [ ]:
def metrics(preds, labels):
    """
    Iceberg = class 1 + class 2 (shadow is part of iceberg).
    Binary IoU: iceberg vs ocean. Area bias on iceberg pixels.
    """
    # merge classes 1 and 2 into one iceberg class
    pred_ice = (preds >= 1)
    true_ice = (labels >= 1)

    tp = (pred_ice & true_ice).sum()
    fp = (pred_ice & ~true_ice).sum()
    fn = (~pred_ice & true_ice).sum()
    denom = tp + fp + fn
    iou = tp / denom if denom > 0 else float('nan')

    true_count = true_ice.sum()
    pred_count = pred_ice.sum()
    area_bias = 100 * (pred_count - true_count) / true_count if true_count > 0 else float('nan')

    return {
        'iou_iceberg':   iou,
        'area_bias_pct': area_bias,
    }

In [ ]:
# sanity check baseline
baseline_m = metrics(baseline_preds, all_labels)
print(f'Baseline IoU (iceberg): {baseline_m["iou_iceberg"]:.4f}')
print(f'Baseline area bias:     {baseline_m["area_bias_pct"]:+.2f}%')

In [ ]:
# --- CRF parameter sets to test ---
# edit this dict and re-run from here

SUBSET = 50   # chips to use per config (None = full test set)

param_sets = {
    'deeplab_defaults': dict(
        sxy_gauss=3,   compat_gauss=3,
        sxy_bilateral=80, srgb_bilateral=5, compat_bilateral=4,
        n_iters=10
    ),
    'tight_spatial': dict(
        sxy_gauss=3,   compat_gauss=3,
        sxy_bilateral=40, srgb_bilateral=5, compat_bilateral=4,
        n_iters=10
    ),
    'wide_spatial': dict(
        sxy_gauss=3,   compat_gauss=3,
        sxy_bilateral=100, srgb_bilateral=5, compat_bilateral=4,
        n_iters=10
    ),
    'high_bilateral_wt': dict(
        sxy_gauss=3,   compat_gauss=3,
        sxy_bilateral=80, srgb_bilateral=5, compat_bilateral=6,
        n_iters=10
    ),
    'loose_appearance': dict(
        sxy_gauss=3,   compat_gauss=3,
        sxy_bilateral=80, srgb_bilateral=3, compat_bilateral=4,
        n_iters=10
    ),
}

print(f'{len(param_sets)} configs, {SUBSET or "all"} chips each')

In [ ]:
# run all configurations
n = len(all_probs) if SUBSET is None else min(SUBSET, len(all_probs))
baseline_sub = metrics(baseline_preds[:n], all_labels[:n])

results = {}
for name, params in param_sets.items():
    print(f'{name} ...', end=' ')
    preds = run_crf_batch(all_probs, X_test, params, subset=SUBSET)
    m = metrics(preds, all_labels[:n])
    results[name] = {'preds': preds, 'metrics': m}
    delta = m['iou_iceberg'] - baseline_sub['iou_iceberg']
    print(f'IoU={m["iou_iceberg"]:.4f} ({delta:+.4f})  area_bias={m["area_bias_pct"]:+.2f}%')

In [ ]:
# summary table
print(f'{"config":<22} {"iou_iceberg":>12} {"area_bias%":>11}')
print('-' * 48)
m = baseline_sub
print(f'{"baseline (no CRF)":<22} {m["iou_iceberg"]:>12.4f} {m["area_bias_pct"]:>+11.2f}')
for name, r in results.items():
    m = r['metrics']
    print(f'{name:<22} {m["iou_iceberg"]:>12.4f} {m["area_bias_pct"]:>+11.2f}')

In [ ]:
# visual comparison -- change idx to browse chips
idx = 0

n_cols = 3 + len(param_sets)   # chip + truth + baseline + each CRF config
fig, axes = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 3.5))

# chip
chip = X_test[idx]
rgb  = np.stack([chip[0], chip[1], chip[2]], axis=-1)
rgb  = np.clip(rgb / (np.percentile(rgb, 98) + 1e-6), 0, 1)
axes[0].imshow(rgb)
axes[0].set_title('chip')

# ground truth
axes[1].imshow(all_labels[idx], cmap='tab10', vmin=0, vmax=2)
axes[1].set_title('ground truth')

# baseline
axes[2].imshow(baseline_preds[idx], cmap='tab10', vmin=0, vmax=2)
axes[2].set_title('baseline')

# each CRF config
for col, (name, r) in enumerate(results.items(), start=3):
    axes[col].imshow(r['preds'][idx], cmap='tab10', vmin=0, vmax=2)
    axes[col].set_title(name, fontsize=8)

for ax in axes:
    ax.axis('off')

plt.suptitle(f'chip {idx}  |  0=ocean  1=iceberg  2=shadow')
plt.tight_layout()
plt.show()

In [ ]:
# once you pick a winner, set SUBSET = None above and re-run to get full-set metrics